# Hybrid CNN–Transformer Model for Short-Term Household Energy Consumption Forecasting

This notebook provides a **research-grade, reproducible pipeline** for forecasting household energy consumption using the UCI Household Electric Power Consumption dataset.

## What this notebook covers
- Rigorous preprocessing (datetime parsing, missing value handling, scaling, feature engineering)
- Sliding-window sequence generation for both:
  - **Single-step forecasting**
  - **Multi-step forecasting (next 3 timesteps)**
- Strong baselines (**LSTM**, **GRU**)
- Proposed **Hybrid Multi-Scale CNN + Transformer Encoder** model
- Controlled training setup (callbacks, fixed seeds, consistent splits)
- Evaluation with **RMSE, MAE, MAPE**, visual diagnostics, and model comparison tables
- Research-oriented interpretation of outcomes

> Designed to be **Google Colab compatible** and easy to adapt for ablation studies and publication-quality experiments.

## 1) Environment Setup and Reproducibility

In [ ]:
# If running on Colab, uncomment and run if needed:
# !pip install -q tensorflow pandas scikit-learn matplotlib seaborn

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dataclasses import dataclass
from typing import Dict, Tuple, List

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


def set_global_seed(seed: int = 42) -> None:
    """Set all relevant random seeds for reproducibility."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_global_seed(42)
print("TensorFlow version:", tf.__version__)

## 2) Data Loading and Preprocessing

The original UCI file uses `;` as delimiter and stores missing entries as `?`. We parse `Date` + `Time` into a single datetime index and engineer calendar features.

In [ ]:
# CONFIGURATION
DATA_PATH = "household_power_consumption.txt"  # Change if needed
TARGET_COL = "Global_active_power"
LOOKBACK = 30
MULTI_STEP_HORIZON = 3

NUMERIC_COLS = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

FEATURE_COLS = NUMERIC_COLS + ["hour", "day_of_week", "month"]


# For demonstration fallback using provided sample snippet
SAMPLE_CSV = """Date;Time;Global_active_power;Global_reactive_power;Voltage;Global_intensity;Sub_metering_1;Sub_metering_2;Sub_metering_3
16/12/2006;17:24:00;4.216;0.418;234.840;18.400;0.000;1.000;17.000
16/12/2006;17:25:00;5.360;0.436;233.630;23.000;0.000;1.000;16.000
16/12/2006;17:26:00;5.374;0.498;233.290;23.000;0.000;2.000;17.000
16/12/2006;17:27:00;5.388;0.502;233.740;23.000;0.000;1.000;17.000
16/12/2006;17:28:00;3.666;0.528;235.680;15.800;0.000;1.000;17.000
16/12/2006;17:29:00;3.520;0.522;235.020;15.000;0.000;2.000;17.000
16/12/2006;17:30:00;3.702;0.520;235.090;15.800;0.000;1.000;17.000
16/12/2006;17:31:00;3.700;0.520;235.220;15.800;0.000;1.000;17.000
16/12/2006;17:32:00;3.668;0.510;233.990;15.800;0.000;1.000;17.000
16/12/2006;17:33:00;3.662;0.510;233.860;15.800;0.000;2.000;16.000
16/12/2006;17:34:00;4.448;0.498;232.860;19.600;0.000;1.000;17.000
16/12/2006;17:35:00;5.412;0.470;232.780;23.200;0.000;1.000;17.000
16/12/2006;17:36:00;5.224;0.478;232.990;22.400;0.000;1.000;16.000"""

In [ ]:
def load_raw_data(data_path: str) -> pd.DataFrame:
    """Load the household power dataset using ';' delimiter and proper NA parsing."""
    if os.path.exists(data_path):
        print(f"Loading dataset from: {data_path}")
        df = pd.read_csv(
            data_path,
            sep=';',
            na_values='?',
            low_memory=False
        )
    else:
        print("WARNING: data file not found. Falling back to provided sample rows.")
        from io import StringIO
        df = pd.read_csv(StringIO(SAMPLE_CSV), sep=';', na_values='?')
    return df


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess raw dataframe: datetime parsing, numeric conversion, NA handling, time features."""
    df = df.copy()

    # Combine Date + Time into datetime
    df['Datetime'] = pd.to_datetime(
        df['Date'] + ' ' + df['Time'],
        format='%d/%m/%Y %H:%M:%S',
        errors='coerce'
    )

    # Drop invalid datetime rows
    df = df.dropna(subset=['Datetime'])

    # Convert numerical columns to float
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)

    # Sort by time and set index
    df = df.sort_values('Datetime').set_index('Datetime')

    # Handle missing values with time interpolation then forward/backward fill
    df[NUMERIC_COLS] = df[NUMERIC_COLS].interpolate(method='time')
    df[NUMERIC_COLS] = df[NUMERIC_COLS].ffill().bfill()

    # Add temporal features
    df['hour'] = df.index.hour.astype(float)
    df['day_of_week'] = df.index.dayofweek.astype(float)
    df['month'] = df.index.month.astype(float)

    return df


raw_df = load_raw_data(DATA_PATH)
proc_df = preprocess_dataframe(raw_df)

print("Processed shape:", proc_df.shape)
print(proc_df[FEATURE_COLS].head())

In [ ]:
# Basic sanity checks
assert TARGET_COL in proc_df.columns, f"Target column {TARGET_COL} missing"
assert proc_df[FEATURE_COLS].isna().sum().sum() == 0, "NaNs remain after preprocessing"

# Quick visualization of target over time
plt.figure(figsize=(14, 4))
plt.plot(proc_df.index, proc_df[TARGET_COL], linewidth=0.7)
plt.title("Global Active Power Over Time")
plt.xlabel("Time")
plt.ylabel("kW")
plt.tight_layout()
plt.show()

## 3) Feature Scaling and Sequence Construction

We split chronologically (70/15/15), fit scalers only on training data, and construct supervised learning windows using a sliding horizon.

In [ ]:
@dataclass
class DataBundle:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    feature_scaler: MinMaxScaler
    target_scaler: MinMaxScaler


def split_by_time(df: pd.DataFrame, train_ratio=0.70, val_ratio=0.15) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    return train_df, val_df, test_df


def fit_scalers(train_df: pd.DataFrame, feature_cols: List[str], target_col: str):
    feature_scaler = MinMaxScaler()
    target_scaler = MinMaxScaler()

    feature_scaler.fit(train_df[feature_cols])
    target_scaler.fit(train_df[[target_col]])

    return feature_scaler, target_scaler


def transform_splits(train_df, val_df, test_df, feature_cols, target_col, feature_scaler, target_scaler):
    X_train_df = pd.DataFrame(feature_scaler.transform(train_df[feature_cols]), columns=feature_cols, index=train_df.index)
    X_val_df = pd.DataFrame(feature_scaler.transform(val_df[feature_cols]), columns=feature_cols, index=val_df.index)
    X_test_df = pd.DataFrame(feature_scaler.transform(test_df[feature_cols]), columns=feature_cols, index=test_df.index)

    y_train = target_scaler.transform(train_df[[target_col]]).flatten()
    y_val = target_scaler.transform(val_df[[target_col]]).flatten()
    y_test = target_scaler.transform(test_df[[target_col]]).flatten()

    return X_train_df, X_val_df, X_test_df, y_train, y_val, y_test


def create_sequences(features: np.ndarray, target: np.ndarray, lookback: int = 30, horizon: int = 1):
    """Create sliding-window sequences for single-step or multi-step forecasting.

    features: shape (T, num_features)
    target: shape (T,)
    output y shape:
      - (N,) when horizon=1
      - (N, horizon) when horizon>1
    """
    X, y = [], []
    max_start = len(features) - lookback - horizon + 1
    for i in range(max_start):
        X.append(features[i:i + lookback])
        if horizon == 1:
            y.append(target[i + lookback])
        else:
            y.append(target[i + lookback:i + lookback + horizon])
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    return X, y


def build_data_bundle(df: pd.DataFrame, lookback: int, horizon: int, feature_cols: List[str], target_col: str) -> DataBundle:
    train_df, val_df, test_df = split_by_time(df)
    feature_scaler, target_scaler = fit_scalers(train_df, feature_cols, target_col)

    X_train_df, X_val_df, X_test_df, y_train_raw, y_val_raw, y_test_raw = transform_splits(
        train_df, val_df, test_df, feature_cols, target_col, feature_scaler, target_scaler
    )

    X_train, y_train = create_sequences(X_train_df.values, y_train_raw, lookback, horizon)
    X_val, y_val = create_sequences(X_val_df.values, y_val_raw, lookback, horizon)
    X_test, y_test = create_sequences(X_test_df.values, y_test_raw, lookback, horizon)

    return DataBundle(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        X_test=X_test, y_test=y_test,
        feature_scaler=feature_scaler,
        target_scaler=target_scaler,
    )


single_data = build_data_bundle(proc_df, lookback=LOOKBACK, horizon=1, feature_cols=FEATURE_COLS, target_col=TARGET_COL)
multi_data = build_data_bundle(proc_df, lookback=LOOKBACK, horizon=MULTI_STEP_HORIZON, feature_cols=FEATURE_COLS, target_col=TARGET_COL)

print("Single-step shapes:", single_data.X_train.shape, single_data.y_train.shape)
print("Multi-step shapes:", multi_data.X_train.shape, multi_data.y_train.shape)

## 4) Model Definitions

We implement two recurrent baselines and a proposed deep hybrid architecture:

- **LSTM baseline**
- **GRU baseline**
- **Hybrid Multi-Scale CNN + Transformer Encoder** with positional encoding, residuals, layer normalization, and feed-forward blocks.

In [ ]:
def build_lstm_model(input_shape: Tuple[int, int], horizon: int = 1) -> tf.keras.Model:
    inputs = layers.Input(shape=input_shape)
    x = layers.LSTM(128, return_sequences=True)(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(64)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(horizon)(x)
    model = models.Model(inputs, outputs, name=f"LSTM_h{horizon}")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model


def build_gru_model(input_shape: Tuple[int, int], horizon: int = 1) -> tf.keras.Model:
    inputs = layers.Input(shape=input_shape)
    x = layers.GRU(128, return_sequences=True)(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.GRU(64)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(horizon)(x)
    model = models.Model(inputs, outputs, name=f"GRU_h{horizon}")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model


class PositionalEncoding(layers.Layer):
    def __init__(self, max_len: int, d_model: int, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model

    def get_config(self):
        config = super().get_config()
        config.update({"max_len": self.max_len, "d_model": self.d_model})
        return config

    def build(self, input_shape):
        position = np.arange(self.max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, self.d_model, 2) * -(np.log(10000.0) / self.d_model))

        pe = np.zeros((self.max_len, self.d_model), dtype=np.float32)
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)

        self.pos_encoding = tf.constant(pe[np.newaxis, ...], dtype=tf.float32)
        super().build(input_shape)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pos_encoding[:, :seq_len, :]


def transformer_encoder_block(x, num_heads: int, key_dim: int, ff_dim: int, dropout: float = 0.1):
    # Multi-head self-attention + residual + layernorm
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout)(x, x)
    attn_out = layers.Dropout(dropout)(attn_out)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

    # Feed-forward + residual + layernorm
    ff_out = layers.Dense(ff_dim, activation='gelu')(x)
    ff_out = layers.Dropout(dropout)(ff_out)
    ff_out = layers.Dense(x.shape[-1])(ff_out)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff_out)
    return x


def build_hybrid_cnn_transformer(input_shape: Tuple[int, int], horizon: int = 1) -> tf.keras.Model:
    inputs = layers.Input(shape=input_shape)

    # Multi-scale CNN branch
    c3 = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(inputs)
    c5 = layers.Conv1D(64, kernel_size=5, padding='same', activation='relu')(inputs)
    c7 = layers.Conv1D(64, kernel_size=7, padding='same', activation='relu')(inputs)

    x = layers.Concatenate(axis=-1)([c3, c5, c7])
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    # Project to transformer dimension
    d_model = 192
    x = layers.Dense(d_model)(x)

    # Positional encoding
    x = PositionalEncoding(max_len=input_shape[0], d_model=d_model)(x)

    # Stacked transformer encoder layers
    for _ in range(3):
        x = transformer_encoder_block(x, num_heads=6, key_dim=32, ff_dim=384, dropout=0.1)

    # Pooling + prediction head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(horizon)(x)

    model = models.Model(inputs, outputs, name=f"HybridCNNTransformer_h{horizon}")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

## 5) Training Utilities

In [ ]:
def get_callbacks(patience_es: int = 6, patience_lr: int = 3):
    return [
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=patience_es,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=patience_lr,
            min_lr=1e-6,
            verbose=1,
        )
    ]


def train_model(model: tf.keras.Model, data: DataBundle, epochs: int = 30, batch_size: int = 64):
    history = model.fit(
        data.X_train, data.y_train,
        validation_data=(data.X_val, data.y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=get_callbacks(),
        verbose=1,
    )
    return history


def invert_scale(y_scaled: np.ndarray, scaler: MinMaxScaler) -> np.ndarray:
    """Invert MinMax scaling for 1D or 2D target arrays."""
    y_scaled = np.asarray(y_scaled)
    if y_scaled.ndim == 1:
        return scaler.inverse_transform(y_scaled.reshape(-1, 1)).flatten()
    elif y_scaled.ndim == 2:
        # For multi-step, invert each horizon column independently using same scaler
        cols = [scaler.inverse_transform(y_scaled[:, i].reshape(-1, 1)).flatten() for i in range(y_scaled.shape[1])]
        return np.stack(cols, axis=1)
    else:
        raise ValueError("y_scaled must be 1D or 2D")


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
    mae = mean_absolute_error(y_true.flatten(), y_pred.flatten())

    eps = 1e-8
    mape = np.mean(np.abs((y_true.flatten() - y_pred.flatten()) / (np.abs(y_true.flatten()) + eps))) * 100

    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}


def plot_loss(history, title: str):
    plt.figure(figsize=(8, 4))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_actual_vs_pred(y_true: np.ndarray, y_pred: np.ndarray, title: str, n_points: int = 300):
    plt.figure(figsize=(12, 4))
    plt.plot(y_true.flatten()[:n_points], label='Actual', linewidth=1.0)
    plt.plot(y_pred.flatten()[:n_points], label='Predicted', linewidth=1.0)
    plt.title(title)
    plt.xlabel('Time step')
    plt.ylabel('Global Active Power')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6) Single-Step Forecasting Experiments (Horizon = 1)

Training all three models under identical split and optimization settings:
- Batch size = **64**
- Epochs = **30**
- Callbacks = **EarlyStopping**, **ReduceLROnPlateau**

In [ ]:
input_shape = single_data.X_train.shape[1:]
print("Input shape:", input_shape)

single_models = {
    "LSTM": build_lstm_model(input_shape, horizon=1),
    "GRU": build_gru_model(input_shape, horizon=1),
    "Hybrid CNN-Transformer": build_hybrid_cnn_transformer(input_shape, horizon=1),
}

single_histories = {}
single_metrics = {}
single_predictions = {}

for name, model in single_models.items():
    print("\n" + "="*80)
    print(f"Training {name} (single-step)")
    print("="*80)
    history = train_model(model, single_data, epochs=30, batch_size=64)
    single_histories[name] = history

    y_pred_scaled = model.predict(single_data.X_test, verbose=0).flatten()
    y_true = invert_scale(single_data.y_test, single_data.target_scaler)
    y_pred = invert_scale(y_pred_scaled, single_data.target_scaler)

    metrics = compute_metrics(y_true, y_pred)
    single_metrics[name] = metrics
    single_predictions[name] = (y_true, y_pred)

    print(f"{name} metrics:", metrics)

In [ ]:
# Visual diagnostics for single-step
for name in single_models.keys():
    plot_loss(single_histories[name], title=f"{name} - Training vs Validation Loss (Single-Step)")
    y_true, y_pred = single_predictions[name]
    plot_actual_vs_pred(y_true, y_pred, title=f"{name} - Actual vs Predicted (Single-Step)")

## 7) Multi-Step Forecasting Experiments (Horizon = 3)

Here each model predicts the next **3 future values** at each inference step.

In [ ]:
input_shape_multi = multi_data.X_train.shape[1:]
print("Multi-step input shape:", input_shape_multi)

multi_models = {
    "LSTM": build_lstm_model(input_shape_multi, horizon=MULTI_STEP_HORIZON),
    "GRU": build_gru_model(input_shape_multi, horizon=MULTI_STEP_HORIZON),
    "Hybrid CNN-Transformer": build_hybrid_cnn_transformer(input_shape_multi, horizon=MULTI_STEP_HORIZON),
}

multi_histories = {}
multi_metrics = {}
multi_predictions = {}

for name, model in multi_models.items():
    print("\n" + "="*80)
    print(f"Training {name} (multi-step horizon={MULTI_STEP_HORIZON})")
    print("="*80)
    history = train_model(model, multi_data, epochs=30, batch_size=64)
    multi_histories[name] = history

    y_pred_scaled = model.predict(multi_data.X_test, verbose=0)
    y_true = invert_scale(multi_data.y_test, multi_data.target_scaler)
    y_pred = invert_scale(y_pred_scaled, multi_data.target_scaler)

    metrics = compute_metrics(y_true, y_pred)
    multi_metrics[name] = metrics
    multi_predictions[name] = (y_true, y_pred)

    print(f"{name} metrics:", metrics)

In [ ]:
# Visual diagnostics for multi-step (flattened for concise plotting)
for name in multi_models.keys():
    plot_loss(multi_histories[name], title=f"{name} - Training vs Validation Loss (Multi-Step)")
    y_true, y_pred = multi_predictions[name]
    plot_actual_vs_pred(y_true, y_pred, title=f"{name} - Actual vs Predicted (Multi-Step, Flattened)")

## 8) Comparative Results Tables

In [ ]:
def metrics_to_df(metrics_dict: Dict[str, Dict[str, float]], setting_name: str) -> pd.DataFrame:
    df = pd.DataFrame(metrics_dict).T.reset_index().rename(columns={'index': 'Model'})
    df.insert(0, 'Setting', setting_name)
    return df[['Setting', 'Model', 'RMSE', 'MAE', 'MAPE']]

single_results_df = metrics_to_df(single_metrics, 'Single-Step (H=1)')
multi_results_df = metrics_to_df(multi_metrics, f'Multi-Step (H={MULTI_STEP_HORIZON})')
comparison_df = pd.concat([single_results_df, multi_results_df], axis=0, ignore_index=True)

print("Model Comparison Table")
display(comparison_df.sort_values(['Setting', 'RMSE']))

## 9) Interpretation and Research Notes

### Suggested interpretation template (edit after running):

1. **Overall ranking**: Identify which model has the lowest RMSE/MAE/MAPE in each setting.
2. **Single-step vs multi-step gap**: Quantify how much error increases from H=1 to H=3.
3. **Architectural insight**:
   - If the hybrid model performs best, this supports combining local temporal pattern extraction (multi-scale CNN) with global dependency modeling (Transformer attention).
   - Compare whether RNN baselines degrade faster in multi-step settings.
4. **Training dynamics**:
   - Discuss convergence behavior from train/validation curves.
   - Mention whether LR reduction and early stopping activated meaningfully.
5. **Practical implication**:
   - Translate improvements into operational value for demand-side management or household-level scheduling.

### Optional enhancements for publication-quality ablation studies
- Remove one CNN kernel branch at a time (3, 5, 7) to test multi-scale contribution.
- Vary number of Transformer layers/heads.
- Add exogenous engineered features (weekend flag, holiday flags, lag statistics).
- Evaluate horizon-specific errors (t+1, t+2, t+3 separately).

## 10) Save Artifacts (Optional)

You can save trained models and result tables for paper appendices or reproducibility packages.

In [ ]:
# Example artifact saving
# for name, model in single_models.items():
#     model.save(f"{name.replace(' ', '_').lower()}_single.keras")
#
# for name, model in multi_models.items():
#     model.save(f"{name.replace(' ', '_').lower()}_multi.keras")
#
# comparison_df.to_csv("model_comparison_results.csv", index=False)

print("Notebook pipeline complete. Run all cells sequentially for full experiments.")